In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
import optuna

from nerf import LNeRF
from utils.rays import intrinsic, create_rays, look_at, equidistance_rotations

COMPUTE_DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    COMPUTE_DEVICE = torch.device('cuda:0')
elif torch.mps.is_available():
    COMPUTE_DEVICE = torch.device('mps')
print(f"{COMPUTE_DEVICE=}")

RUN_OPT = False

# NeRF to point cloud evaluation

## Setup

In [ ]:
def get_origin_direction_eq_angles(n, img_shape, focal):
    phis, thetas = equidistance_rotations(n)

    origin, direction = [], []
    for theta, phi in zip(thetas, phis):
        o, d = create_rays(
            img_shape[0],
            img_shape[1],
            intrinsic(focal, img_shape),
            look_at(4, theta, phi)
        )
        origin.append(o)
        direction.append(d)

    return torch.stack(origin), torch.stack(direction)

In [ ]:
model = LNeRF.load_from_checkpoint(
    "../lightning_logs/shurtape200x200_decay=1e-06_exp/checkpoints/best_val_psnr_epoch=9.ckpt",
    map_location=COMPUTE_DEVICE,
    hparams_file="../lightning_logs/shurtape200x200_decay=1e-06_exp/hparams.yaml"
)
model.freeze()
model.eval()
focal = torch.tensor(model.hparams.focal, dtype=torch.float32, device=COMPUTE_DEVICE)

origin, direction = get_origin_direction_eq_angles(8, (200, 200), focal.expand(2))
print(f"{origin.shape=} {direction.shape=}")

In [ ]:
depth, sp_mask = model.locate_density_gradient_based_surface_depth(
    origin, direction,
    sigma_limit=3.5,
    gamma=7e-8,
    non_grad_step_size=2.4e-2,
    grad_epsilon=6.4e-3,
    max_iters=300
)
depth, sp_mask = depth.detach().cpu(), sp_mask.cpu()
print(f"{depth.shape=} {sp_mask.shape=}")

In [ ]:
depth_imgs = (depth.expand((-1, -1, -1, 3)) - model.hparams.near) / (model.hparams.far - model.hparams.near)
depth_grid = make_grid(depth_imgs.permute(0, 3, 1, 2), 4).permute(1, 2, 0).clip(0,1)

fig, ax = plt.subplots(1, 1, figsize=(18, 6))
ax.imshow(depth_grid)
ax.axis('off')
plt.show()

In [ ]:
depth_mask = ((depth > model.hparams.near) & (depth <= model.hparams.far)).squeeze(-1)
points = origin[depth_mask] + depth[depth_mask] * direction[depth_mask]

points.shape, points[sp_mask[depth_mask]].shape

## Chamfer Distance

In [ ]:
cloud_from_tensor = lambda tens: o3d.geometry.PointCloud(o3d.utility.Vector3dVector(tens.cpu()))

def load_and_normalize_mesh(obj_path: Path) -> o3d.geometry.TriangleMesh:
    mesh = o3d.io.read_triangle_mesh(obj_path)
    bbox = mesh.get_axis_aligned_bounding_box()
    # Re-centering and scaling to -1:1 bounding box, confirmed same transforms to Mitsuba version
    scale = 1 / max((bbox.get_max_bound() - bbox.get_min_bound()) / 2)
    translate = -bbox.get_center()

    mesh = mesh.translate(translate).scale(scale.item(), [0, 0, 0])
    return mesh

def mesh_to_cloud_signed_distances(mesh: o3d.geometry.TriangleMesh, cloud: o3d.geometry.PointCloud) -> np.ndarray:
    cloud = o3d.t.geometry.PointCloud.from_legacy(cloud)
    mesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
    scene = o3d.t.geometry.RaycastingScene()
    _ = scene.add_triangles(mesh)
    sdf = scene.compute_signed_distance(cloud.point.positions).abs()
    return sdf.numpy()

In [ ]:
mesh = load_and_normalize_mesh(Path("../data/raw_objects/Shurtape_Tape_Purple_CP28/meshes/model.obj"))
point_cloud = cloud_from_tensor(points)

print(mesh, point_cloud, sep='\n')

In [ ]:
def colored_boxplot(vals, colormap):
    # Create horizontal boxplot
    fig, ax = plt.subplots(figsize=(14, 2))
    bp = ax.boxplot(vals, vert=False, widths=1.0)

    for element in ['boxes', 'whiskers', 'caps']:
        plt.setp(bp[element], color="lime", linewidth=1.5)
    plt.setp(bp['fliers'], color="lime", linewidth=1, markeredgecolor="lime")
    plt.setp(bp['medians'], color="white", linewidth=3)

    # Color the background based on distance distribution
    xmin, xmax = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    # Create a gradient background
    gradient = np.linspace(xmin, xmax, 256)
    gradient = np.vstack((gradient, gradient))
    extent = (xmin, xmax, y_min - 1, y_max + 1)  # Extend beyond boxplot bounds

    # Apply the colormap to the background
    ax.imshow(
        gradient, 
        aspect='auto', 
        extent=extent, 
        cmap=colormap, 
        alpha=1.0, 
        origin='lower'
    )

    # Add mean line
    mean = vals.mean()
    ax.axvline(mean, color="grey", linestyle='--', linewidth=2)
    ax.text(mean, 2.0, f'Mean: {mean:.4f}', ha='center', color="grey", fontdict={'weight': 'bold'})

    ax.set_yticks([])  # Hide y-axis as it's just one boxplot
    ax.set_xlabel('Chamfer Distance')
    plt.tight_layout()
    plt.show()


In [ ]:
point_cloud = cloud_from_tensor(points)
# Remove obvious outliers, results of numerical errors and local maxima above threshold in specific points
point_cloud, _ = point_cloud.remove_statistical_outlier(nb_neighbors=10, std_ratio=5.0)
mesh = load_and_normalize_mesh(Path("../data/raw_objects/Shurtape_Tape_Purple_CP28/meshes/model.obj"))

print(point_cloud)
cd = mesh_to_cloud_signed_distances(mesh, point_cloud)

colormap = plt.get_cmap('seismic')
colored_boxplot(cd, colormap)

rgb = colormap((cd - cd.min()) / (cd.max() - cd.min()))[:, :3]
point_cloud.colors = o3d.utility.Vector3dVector(rgb)
o3d.visualization.draw_geometries([
    point_cloud
])

## Parameter serach function

In [ ]:
def raymeshnerf_cloud_chamfer_distance(trial: optuna.Trial):
    model = LNeRF.load_from_checkpoint(
        "../lightning_logs/shurtape200x200_decay=1e-06_exp/checkpoints/best_val_psnr_epoch=9.ckpt",
        map_location=COMPUTE_DEVICE,
        hparams_file="../lightning_logs/shurtape200x200_decay=1e-06_exp/hparams.yaml"
    )
    model.freeze()
    model.eval()
    focal = torch.tensor(model.hparams.focal, dtype=torch.float32, device=COMPUTE_DEVICE)
    
    origin, direction = get_origin_direction_eq_angles(8, (200, 200), focal.expand(2))

    depth, sp_mask = model.locate_density_gradient_based_surface_depth(
        origin, direction,
        sigma_limit=trial.suggest_float("sigma_limit", 2.0, 8.0),  # base: 5.0
        gamma=trial.suggest_float("gamma", 1e-8, 1e-5),  # base: 5e-6
        non_grad_step_size=trial.suggest_float("non_grad_step_size", 5e-3, 5e-2),  # base:3e-2
        grad_epsilon=trial.suggest_float("grad_epsilon", 1e-4, 1e-2), # base: 5e-2
        max_iters=100 * trial.suggest_int("max_iters/100", 1, 5),
    )
    depth, sp_mask = depth.detach().cpu(), sp_mask.cpu()

    depth_mask = ((depth > model.hparams.near) & (depth <= model.hparams.far)).squeeze(-1)
    points = origin[depth_mask] + depth[depth_mask] * direction[depth_mask]

    point_cloud = cloud_from_tensor(points)
    # Remove obvious outliers, results of numerical errors and local maxima above threshold in specific points
    point_cloud, _ = point_cloud.remove_statistical_outlier(nb_neighbors=10, std_ratio=5.0)

    # Setting verbosity level to Error as mesh doesn't have texture next to it and loads as multiple materials,
    #   this gives warnings but doesn't matter for eval 
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)
    mesh = load_and_normalize_mesh(Path("../data/raw_objects/Shurtape_Tape_Purple_CP28/meshes/model.obj").resolve())

    cd = mesh_to_cloud_signed_distances(mesh, point_cloud)
    # Reset verbosity level to recieve info about potential other problems
    o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Info)

    trial.set_user_attr("point_count", len(point_cloud.points))
    trial.set_user_attr("finished_points", int(sp_mask.sum().item()))
    return cd.mean().item()


if RUN_OPT:
    study = optuna.create_study(
        study_name="ShurtapeChamferDistance",
        direction="minimize",
        storage="sqlite:///ShurtapeCD.db"
    )

    study.optimize(
        raymeshnerf_cloud_chamfer_distance,
        n_trials=100,
        gc_after_trial=True
    )